In [4]:
import os
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(Path.cwd())

/Users/majdalsuleh/Projects/mining-quality-prediction


In [10]:
con = duckdb.connect("data/mining.duckdb", read_only=True)
hourly = con.sql("""
    SELECT * FROM hourly
    WHERE hour >= TIMESTAMP '2017-03-29 12:00:00'
    ORDER BY hour
""").df()
con.close()

hourly = hourly.set_index("hour")
hourly.shape

(3948, 25)

In [11]:
hourly.index.to_series().diff().value_counts()

hour
0 days 01:00:00    3947
Name: count, dtype: int64

In [12]:
hourly["silica_known"] = (
    hourly["silica_conc"]
    .where(~hourly["is_filled"])   # hide filled-in values
    .ffill()                       # carry the last real value forward
)

In [13]:
hourly["target"] = hourly["silica_conc"].shift(-1)
hourly["target_is_real"] = ~hourly["is_filled"].shift(-1, fill_value=True)

print(hourly["target_is_real"].sum(), "hours with a real next-hour target")

3508 hours with a real next-hour target
